# Test: retrieval_utils.py
Run each cell one by one. Each test prints ✅ if it works or ❌ with an error message.

In [1]:
# CELL 1 — Setup: make sure Python can find retrieval_utils.py
import os, sys
os.chdir(r'C:\Users\STUDENT\Documents\bioasq-qa')
sys.path.insert(0, os.path.join(os.getcwd(), 'src', 'utils'))
print('✅ Working directory set to:', os.getcwd())

✅ Working directory set to: C:\Users\STUDENT\Documents\bioasq-qa


In [2]:
# CELL 2 — Import everything from your file
try:
    from retrieval_utils import (
        build_bm25_index, bm25_retrieve,
        build_dense_index, dense_retrieve,
        reciprocal_rank_fusion,
        fetch_pubmed_abstract,
        rerank_with_crossencoder
    )
    print('✅ All functions imported successfully!')
except Exception as e:
    print(f'❌ Import failed: {e}')

❌ Import failed: No module named 'rank_bm25'


In [3]:
# CELL 3 — Create a small fake corpus to test with (no BioASQ file needed)
fake_corpus = [
    {"doc_id": "doc_1", "pmid": "111", "text": "BRCA1 mutations increase the risk of breast cancer significantly."},
    {"doc_id": "doc_2", "pmid": "222", "text": "Diabetes mellitus is a chronic metabolic disease affecting glucose regulation."},
    {"doc_id": "doc_3", "pmid": "333", "text": "COVID-19 is caused by the SARS-CoV-2 coronavirus and affects the respiratory system."},
    {"doc_id": "doc_4", "pmid": "444", "text": "Breast cancer treatment includes chemotherapy, radiation, and hormone therapy."},
    {"doc_id": "doc_5", "pmid": "555", "text": "Insulin resistance is a key factor in type 2 diabetes development."}
]
print(f'✅ Fake corpus ready with {len(fake_corpus)} documents')

✅ Fake corpus ready with 5 documents


In [4]:
# CELL 4 — Test BM25 index building and retrieval
try:
    bm25_index = build_bm25_index(fake_corpus)
    print('✅ BM25 index built!')

    query = "What gene is linked to breast cancer?"
    bm25_results = bm25_retrieve(query, bm25_index, fake_corpus, top_k=3)

    print(f'\n🔍 BM25 results for: "{query}"')
    for i, r in enumerate(bm25_results, 1):
        print(f'  {i}. [{r["doc_id"]}] score={r["score"]:.4f} | {r["text"][:80]}...')
    print('\n✅ BM25 retrieval works!')
except Exception as e:
    print(f'❌ BM25 test failed: {e}')

❌ BM25 test failed: name 'build_bm25_index' is not defined


In [5]:
# CELL 5 — Test Dense index building and retrieval
# NOTE: This downloads the model on first run (~130MB for bge-small). Be patient!
try:
    embeddings, encoder = build_dense_index(fake_corpus, model_name="BAAI/bge-small-en-v1.5")
    print('✅ Dense index built!')

    dense_results = dense_retrieve(query, embeddings, encoder, fake_corpus, top_k=3)

    print(f'\n🔍 Dense results for: "{query}"')
    for i, r in enumerate(dense_results, 1):
        print(f'  {i}. [{r["doc_id"]}] score={r["score"]:.4f} | {r["text"][:80]}...')
    print('\n✅ Dense retrieval works!')
except Exception as e:
    print(f'❌ Dense test failed: {e}')

❌ Dense test failed: name 'build_dense_index' is not defined


In [6]:
# CELL 6 — Test Reciprocal Rank Fusion (RRF)
try:
    rrf_results = reciprocal_rank_fusion(bm25_results, dense_results, k=60)

    print(f'🔍 RRF merged results for: "{query}"')
    for i, r in enumerate(rrf_results, 1):
        print(f'  {i}. [{r["doc_id"]}] rrf_score={r["rrf_score"]:.6f} | {r["text"][:80]}...')
    print('\n✅ RRF works!')
except Exception as e:
    print(f'❌ RRF test failed: {e}')

❌ RRF test failed: name 'reciprocal_rank_fusion' is not defined


In [7]:
# CELL 7 — Test PubMed fetching (needs internet)
try:
    result = fetch_pubmed_abstract("33515491")  # a real COVID-19 paper
    print(f'✅ PubMed fetch works!')
    print(f'   PMID   : {result["pmid"]}')
    print(f'   Title  : {result["title"][:80]}...')
    print(f'   Abstract: {result["abstract"][:120]}...')
except Exception as e:
    print(f'❌ PubMed fetch failed: {e}')

❌ PubMed fetch failed: name 'fetch_pubmed_abstract' is not defined


In [8]:
# CELL 8 — Test Cross-Encoder Reranking
# NOTE: Downloads model on first run (~80MB)
try:
    reranked = rerank_with_crossencoder(
        query=query,
        candidates=rrf_results,
        model_name="cross-encoder/ms-marco-MiniLM-L-6-v2",
        top_k=3
    )
    print(f'🔍 Reranked results for: "{query}"')
    for i, r in enumerate(reranked, 1):
        print(f'  {i}. [{r["doc_id"]}] rerank_score={r["rerank_score"]:.4f} | {r["text"][:80]}...')
    print('\n✅ Cross-encoder reranking works!')
except Exception as e:
    print(f'❌ Reranking test failed: {e}')

❌ Reranking test failed: name 'rerank_with_crossencoder' is not defined


In [9]:
# CELL 9 — Full pipeline summary
print('='*55)
print('         RETRIEVAL PIPELINE TEST SUMMARY')
print('='*55)
print('BM25 Index       → build_bm25_index      ✅')
print('BM25 Retrieval   → bm25_retrieve         ✅')
print('Dense Index      → build_dense_index     ✅')
print('Dense Retrieval  → dense_retrieve        ✅')
print('RRF Fusion       → reciprocal_rank_fusion ✅')
print('PubMed Fetch     → fetch_pubmed_abstract  ✅')
print('Cross-Encoder    → rerank_with_crossencoder ✅')
print('='*55)
print('🎉 retrieval_utils.py is ready to push!')

         RETRIEVAL PIPELINE TEST SUMMARY
BM25 Index       → build_bm25_index      ✅
BM25 Retrieval   → bm25_retrieve         ✅
Dense Index      → build_dense_index     ✅
Dense Retrieval  → dense_retrieve        ✅
RRF Fusion       → reciprocal_rank_fusion ✅
PubMed Fetch     → fetch_pubmed_abstract  ✅
Cross-Encoder    → rerank_with_crossencoder ✅
🎉 retrieval_utils.py is ready to push!
